This implementation builds on the GQA + XSA model by replacing the dense FFN in each transformer block with a Mixture of Experts (MoE) layer. Each token is routed to the top-K experts out of N, with the output being a weighted sum of the selected experts' outputs — giving the model more capacity without proportionally increasing compute per token.

In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
from transformers import AutoTokenizer
import numpy as np
import json
import os
from llm_module import (
    ModelMoE, LMDataset, generate_sample, train, calculate_loader_loss,
    calculate_perplexity, generate, generate_text_stream_cache, KVCache
)

Similar to my other implementation, I will use the RedPajama dataset and the gpt-2 tokenizer to keep the vocab size limited due to hardware limitations.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')

In [ ]:
# Reusing local data generated by original notebook
train_path = 'train.bin'
val_path = 'val.bin'
test_path = 'test.bin'

CONTEXT_LENGTH = 512

train_dataset = LMDataset(train_path, CONTEXT_LENGTH)
val_dataset = LMDataset(val_path, CONTEXT_LENGTH)
test_dataset = LMDataset(test_path, CONTEXT_LENGTH)

In [ ]:
# Test dataset
print(train_dataset.tokens[:10])

In [ ]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [ ]:
# Test dataloader
example = next(iter(train_loader))
print(example)

## Prepare the model

This implementation replaces the dense FFN in each transformer block with a Mixture of Experts layer. The router sends each token to the top-K experts; outputs are weighted by softmax scores over the selected experts.

#### Initialize the model

In [ ]:
MODEL_CONFIG = {
    'vocab_size': 50257,
    'context_length': CONTEXT_LENGTH,
    'n_layers': 13,
    'n_heads': 8,
    'n_kv_groups': 4,
    'emb_dim': 768,
    'hidden_dim': 2048,
    'n_experts': 8,
    'n_experts_token': 2,
    'dtype': torch.bfloat16
}

model = ModelMoE(MODEL_CONFIG, xsa=False)

In [ ]:
# Using mps, modify for cuda
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

print('Device:', device)
model.to(device)

In [ ]:
parameters = sum(p.numel() for p in model.parameters())
print(f'Parameters: {parameters:,}')

In [ ]:
model = torch.compile(model, dynamic=True)

### Training

In [ ]:
# Test model forward passes and generate function
generate_sample(
    model=model,
    tokenizer=tokenizer,
    device=device,
    context='Dedication will always pay',
    context_size=MODEL_CONFIG['context_length']
)

In [ ]:
torch.manual_seed(123)

num_epochs = 1
initial_lr = 1e-5
min_lr = 4e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, fused=True)

# 5% warmup
warmup_steps = int((len(train_loader) * num_epochs) * 0.05)

# Improve training time
GRAD_ACCUM_STEPS = 2

start_training_time = time.time()
train_losses, val_losses, train_perplexities, val_perplexities, tokens_seen = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=num_epochs,
    optimizer=optimizer,
    eval_freq=10000,
    eval_batches=10,
    message_freq=20000,
    tokenizer=tokenizer,
    device=device,
    warmup_steps=warmup_steps,
    initial_lr=initial_lr,
    min_lr=min_lr,
    model_name='moe.pth',
    grad_accum_steps=GRAD_ACCUM_STEPS
)
end_training_time = time.time()

In [ ]:
torch.save({
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
}, 'moe.pth')

In [ ]:
print(f'Elapsed time: {(end_training_time-start_training_time)/60:.2f} minutes')

### Evaluation

In [ ]:
steps = np.arange(0, len(train_losses) * 10000, 10000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(steps, train_losses, label='Train')
ax1.plot(steps, val_losses, label='Validation')
ax1.set_xlabel('Steps')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over training steps')
ax1.legend()

ax2.plot(steps, train_perplexities, label='Train')
ax2.plot(steps, val_perplexities, label='Validation')
ax2.set_xlabel('Steps')
ax2.set_ylabel('Perplexity')
ax2.set_title('Perplexity over training steps')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    test_loss = calculate_loader_loss(
        model=model,
        loader=test_loader,
        eval_batches=10,
        device=device
    )

print(f'Test loss: {test_loss:.3f}')

test_perplexity = calculate_perplexity(test_loss)
print(f'Test perplexity: {test_perplexity:.2f}')

In [ ]:
text1 = 'Dedication will always pay'
text2 = 'Large language models learn by'

print('Generated from example 1:\n')
generated_tokens_1 = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=text1,
    device=device,
    verbose=True,
    max_new_tokens=50,
    temperature=1.2,
    top_k=20
)
print('\n\n\nGenerated from example 2:\n')
generated_tokens_2 = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=text2,
    device=device,
    verbose=True,
    max_new_tokens=50,
    temperature=1.2,
    top_k=20
)

In [ ]:
metrics = {
    "model": "moe",
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_perplexities": [p.item() if hasattr(p, "item") else p for p in train_perplexities],
    "val_perplexities": [p.item() if hasattr(p, "item") else p for p in val_perplexities],
}

metrics_path = "moe-metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to {os.path.abspath(metrics_path)}")